# Lab 11 — Agentic SDLC Factory

## Objective

Turn a requirement into a planned change, test it, revise it and stage it locally.

## Introduction and concept

The initial fixed implementation treats True as an integer. The failing test triggers a revised template and all cases must pass before staging.

## Real-world use case

A duration validator must reject booleans as well as zero, floats and negative values.

## Technology stack and dependencies

Python compile, tempfile and pathlib. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Requirement → Plan → Template generation → Test → Revise → Review → Local staging


## Step 1 — Implement the components

The initial fixed implementation treats True as an integer. The failing test triggers a revised template and all cases must pass before staging.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

REQUIREMENT = "Accept positive whole-minute durations and reject floats, booleans and zero."
PLAN = ["validate input", "test accepted and rejected examples", "review", "stage artifact"]
DRAFT = "def valid(minutes):\n    return isinstance(minutes, int) and minutes > 0\n"
REVISION = "def valid(minutes):\n    return type(minutes) is int and minutes > 0\n"

def evaluate(source):
    namespace = {}
    # These are fixed templates defined above, never user or model text.
    exec(compile(source, "<fixed-duration-template>", "exec"), namespace)
    cases = [(15, True), (0, False), (-1, False), (2.5, False), (True, False)]
    return [repr(value) for value, expected in cases if namespace["valid"](value) != expected]

def factory(staging_dir):
    trace = [("requirements", REQUIREMENT), ("plan", PLAN)]
    trace.append(("generate", "fixed first-pass template"))
    failures = evaluate(DRAFT)
    trace.append(("test_feedback", failures))
    if failures != ["True"]:
        raise RuntimeError("Unexpected initial test result")
    trace.append(("revise", "use exact integer type"))
    remaining = evaluate(REVISION)
    trace.append(("retest", remaining))
    if remaining:
        return trace
    trace.append(("review", "fixture checklist passed"))
    staged = staging_dir / "duration_rule.py"
    staged.write_text(REVISION, encoding="utf8")
    trace.append(("local_deploy", staged.name))
    return trace

## Step 2 — Run sample inputs and inspect actual output

Use a duration requirement that excludes boolean values. Review the failed assertion case, revised implementation and staged artifact.

In [2]:
def main():
    with TemporaryDirectory() as folder:
        staging_dir = Path(folder)
        trace = factory(staging_dir)
        for stage, outcome in trace:
            print(stage, outcome)
        assert trace[-1][0] == "local_deploy"
        assert evaluate((staging_dir / "duration_rule.py").read_text(encoding="utf8")) == []
        print("PASS: test feedback fixed regression and gated staging")

main()

requirements Accept positive whole-minute durations and reject floats, booleans and zero.
plan ['validate input', 'test accepted and rejected examples', 'review', 'stage artifact']
generate fixed first-pass template
test_feedback ['True']
revise use exact integer type
retest []
review fixture checklist passed
local_deploy duration_rule.py
PASS: test feedback fixed regression and gated staging


## Testing and validation

Use a duration requirement that excludes boolean values. Review the failed assertion case, revised implementation and staged artifact. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

This is fixed-template generation with a local staging artifact. It does not execute user-provided source or deploy to a cloud service.

## How this relates to ExamPilot

Partial. ExamPilot includes component and workflow tests such as backend/app/graph/test_full_exam_graph.py. The full SDLC agent loop is a lab extension.

## Conclusion

Turn a requirement into a planned change, test it, revise it and stage it locally. Use the trace and assertions to explain which decisions the workflow makes.